Tomado de:
A Quick Introduction to PyTorch: Using Deep Learning for Stock Price Prediction
Published: February 23, 2022
5 min read
Written by: H2O.ai Team

https://h2o.ai/blog/2022/a-quick-introduction-to-pytorch-using-deep-learning-for-stock-price-prediction/$0

In [0]:
!pip install statsmodels
!pip install linearmodels
!pip install lseg.data
!pip install pandas_datareader
!pip install torch

In [0]:
import lseg.data as ld
import numpy as np
import pandas as pd

from datetime import datetime, timedelta

In [0]:
# Fetch Apple stock prices for the last year using ld API
# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")

In [0]:
# Calculate date range for last year
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

# Fetch historical data using ld.get_history (not get_data)
df = ld.get_history(

    universe="AAPL.O",
    fields=["TR.PriceClose.date", "TR.PriceClose", "TR.PriceOpen", "TR.PriceHigh", "TR.PriceLow", "TR.Volume"],
    interval="1D",
    start=start_date,
    end=end_date
)

# Display the result
display(df.head())

In [0]:
# Rename columns to standard names
df = df.rename(columns={
    'Price Close': 'close',
    'Price Open': 'open',
    'Price High': 'high',
    'Price Low': 'low'
})

display(df.head())

In [0]:
import matplotlib.pyplot as plt

plt.plot(df.open.values, color='red', label='open')
plt.plot(df.close.values, color='green', label='close')
plt.plot(df.low.values, color='blue', label='low')
plt.plot(df.high.values, color='black', label='high')
plt.title('stock price')
plt.xlabel('time [days]')
plt.ylabel('price')
plt.legend(loc='best')

In [0]:
import sklearn.preprocessing

min_max_scaler = sklearn.preprocessing.MinMaxScaler()

df['open'] = min_max_scaler.fit_transform(df.open.values.reshape(-1,1))
df['high'] = min_max_scaler.fit_transform(df.high.values.reshape(-1,1))
df['low'] = min_max_scaler.fit_transform(df.low.values.reshape(-1,1))
df['close'] = min_max_scaler.fit_transform(df['close'].values.reshape(-1,1))
data = df[['open','close','low','high']].values

In [0]:
data

In [0]:
seq_len=20
sequences=[]
for index in range(len(data) - seq_len): 
 sequences.append(data[index: index + seq_len])
sequences= np.array(sequences)

In [0]:
sequences

In [0]:
valid_set_size_percentage = 10 
test_set_size_percentage = 10 

valid_set_size = int(np.round(valid_set_size_percentage/100*sequences.shape[0])) 
test_set_size = int(np.round(test_set_size_percentage/100*sequences.shape[0]))
train_set_size = sequences.shape[0] - (valid_set_size + test_set_size)

x_train = sequences[:train_set_size,:-1,:]
y_train = sequences[:train_set_size,-1,:]

x_valid = sequences[train_set_size:train_set_size+valid_set_size,:-1,:]
y_valid = sequences[train_set_size:train_set_size+valid_set_size,-1,:]

x_test = sequences[train_set_size+valid_set_size:,:-1,:]
y_test = sequences[train_set_size+valid_set_size:,-1,:]

In [0]:
import torch
from torch.utils.data import TensorDataset, DataLoader

x_train = torch.tensor(x_train).float()
y_train = torch.tensor(y_train).float()

x_valid = torch.tensor(x_valid).float()
y_valid = torch.tensor(y_valid).float()

train_dataset = TensorDataset(x_train,y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(x_valid,y_valid)
valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=True)

In [0]:
from torch import nn

class NeuralNetwork(nn.Module):
  def __init__(self):
    super(NeuralNetwork, self).__init__()
    self.lstm = nn.LSTM(4,64,batch_first=True)
    self.fc = nn.Linear(64,4)

  def forward(self, x):
    output, (hidden, cell) = self.lstm(x)
    x = self.fc(hidden)
    return x
 
model = NeuralNetwork()

#push to cuda if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
model = model.to(device)

In [0]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters())
mse = nn.MSELoss()

In [0]:
def train(dataloader):
  epoch_loss = 0
  model.train() 

  for batch in dataloader:
    optimizer.zero_grad() 
    x,y= batch
    pred = model(x)
    loss = mse(pred[0],y) 
    loss.backward() 
    optimizer.step() 
    epoch_loss += loss.item() 
  return epoch_loss

In [0]:
def evaluate(dataloader):
  epoch_loss = 0
  model.eval() 

  with torch.no_grad():
    for batch in dataloader: 
      x,y= batch
      pred = model(x)
      loss = mse(pred[0],y) 
      epoch_loss += loss.item() 
  return epoch_loss / len(dataloader)

In [0]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x_train = x_train.to(device)
y_train = y_train.to(device)
x_valid = x_valid.to(device)
y_valid = y_valid.to(device)

train_dataset = TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(x_valid, y_valid)
valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=True)

n_epochs = 50
best_valid_loss = float('inf')

for epoch in range(n_epochs):
  train_loss = train(train_dataloader)
  valid_loss = evaluate(valid_dataloader)

  #save the best model
  if valid_loss < best_valid_loss:
    best_valid_loss = valid_loss
    torch.save(model, 'saved_weights.pt')
    print("Epoch ",epoch+1)
  print(f'\tTrain Loss: {train_loss:.5f}')
print(f'\tVal Loss: {valid_loss:.5f}\n')

In [0]:
model=torch.load('saved_weights.pt', weights_only=False)

In [0]:
x_test= torch.tensor(x_test).float()
x_test = x_test.to(next(model.parameters()).device)

with torch.no_grad():
  y_test_pred = model(x_test)

y_test_pred = y_test_pred.cpu().numpy()[0]

In [0]:
idx=0

plt.plot(np.arange(y_train.shape[0], y_train.shape[0]+y_test.shape[0]),
 y_test[:,idx], color='black', label='test target')

plt.plot(np.arange(y_train.shape[0], y_train.shape[0]+y_test_pred.shape[0]),
 y_test_pred[:,idx], color='green', label='test prediction')

plt.title('future stock prices')
plt.xlabel('time [days]')
plt.ylabel('normalized price')
plt.legend(loc='best')